<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 25px; text-align: center; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.2); margin: 20px 0;">

# Demo.3 - 人像分割与背景替换

本 Notebook 演示如何使用 MediaPipe **Selfie Segmenter** 实现实时人像抠图，并支持三种背景模式切换。

</div>

<div style="background-color: #eef6ff; border-left: 4px solid #2563eb; padding: 15px; border-radius: 4px; margin-top: 12px;">

## 本节你将学会

1. 理解分割模型输出的不是类别标签，而是逐像素置信度掩码。
2. 理解 Alpha 合成为什么能实现自然的边缘过渡。
3. 比较绿色背景、自定义背景和原始画面三种模式的视觉差异。

**建议课堂观察**：人物边缘、头发丝、手指缝这些位置最能体现分割质量。

</div>

### 课堂观察任务

这个实验最重要的不是“背景换掉了”，而是学会看**分割边缘质量**。运行前请先明确观察重点：

1. 哪些区域最容易分割失败：头发丝、手指缝、衣服边缘，还是快速移动的部位？
2. 为什么同一张人像，换成不同背景后，边缘问题会变得更明显或更不明显？
3. 绿色背景、自定义图片背景、原始画面三种模式，分别更适合观察什么问题？

建议同学重点盯住人物轮廓边界，而不是只看画面整体是否“好看”。

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 0. 环境安装

首次运行，请执行以下安装命令

</div>

In [ ]:
# %pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python opencv-contrib-python-headless
%pip install mediapipe opencv-python numpy

### 安装后必须执行这一步

如果上面的安装单元执行完成，请先**重启内核**，再从“导入依赖库”开始重新运行。

否则 `cv2.imshow` 相关问题可能不会立刻消失，因为当前内核还在使用旧版 OpenCV。

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 1. 导入依赖库

</div>

In [1]:
from pathlib import Path
import time

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision
import numpy as np

print("依赖库导入成功 ✓")

依赖库导入成功 ✓


<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 2. 常量配置

- 背景图片路径 `BACKGROUND_IMAGE_PATH`：可替换为 `assets/` 下的任意图片
- 分割模型使用 `.tflite` 格式，比手部模型更轻量

</div>

In [2]:
WINDOW_NAME = "Demo.3 - Segmentation Green Screen"
MODEL_PATH = Path("models/selfie_segmenter.tflite")
BACKGROUND_IMAGE_PATH = Path("assets/p2.jpg")  # 可替换为你的背景图

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"未找到人像分割模型文件: {MODEL_PATH}")

print(f"背景图路径: {BACKGROUND_IMAGE_PATH}")
print(f"模型路径:   {MODEL_PATH} ✓")
if not BACKGROUND_IMAGE_PATH.exists():
    print("提示：未找到背景图片，将自动生成渐变色占位背景。")

背景图路径: assets\p2.jpg
模型路径:   models\selfie_segmenter.tflite ✓


<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 3. 背景加载与 Alpha 合成

若背景图不存在，自动生成渐变色占位背景。Alpha 合成公式：

$$output = foreground \times \alpha + background \times (1 - \alpha)$$

</div>

In [3]:
def load_background(target_size):
    """加载背景图片，若不存在则生成渐变占位背景。"""
    path = Path(BACKGROUND_IMAGE_PATH)
    if path.exists():
        image = cv2.imread(str(path))
        if image is not None:
            return cv2.resize(image, target_size, interpolation=cv2.INTER_LINEAR)
    # 生成渐变色占位背景
    w, h = target_size
    x = np.linspace(0, 1, w, dtype=np.float32)
    y = np.linspace(0, 1, h, dtype=np.float32)
    xv, yv = np.meshgrid(x, y)
    bg = np.zeros((h, w, 3), dtype=np.uint8)
    bg[..., 0] = (80 + 120 * xv).astype(np.uint8)
    bg[..., 1] = (60 + 140 * yv).astype(np.uint8)
    bg[..., 2] = (180 - 70 * xv).astype(np.uint8)
    return bg


def composite(frame, mask, background):
    """将前景（人像）与背景按 mask 权重混合。mask 值域 [0,1]，1=人像，0=背景。"""
    alpha = np.clip(mask, 0.0, 1.0)[..., None]
    return (frame * alpha + background * (1.0 - alpha)).astype(np.uint8)

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: white; padding: 8px 20px; display: inline-block; border-radius: 4px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); margin: 30px 0;">

## 4. 主程序：分割与背景替换

### 关键步骤
1. **摄像头采集**：`cv2.VideoCapture(0)` 打开默认摄像头。
2. **人像分割**：`segment_for_video` 获取置信度掩码。
3. **掩码处理**：高斯模糊软化边缘，再用阈值过滤去噪。
4. **Alpha 合成**：按掩码权重混合前景与背景。

### 背景模式切换
| 按键 | 模式 | 说明 |
|------|------|------|
| `x`  | green | 纯绿色背景（绿幕效果）|
| `c`  | custom | 自定义背景图片 |
| `z`  | original | 原始摄像头画面 |
| `q`/`ESC` | — | 退出 |

### 课堂上建议同学重点观察
1. 哪些位置最容易分错：头发边缘、手指缝还是衣服边界？
2. 切换不同背景后，哪些边缘瑕疵会更明显？
3. 把阈值从 0.45 改大或改小，会带来什么变化？

**运行后**：会弹出 OpenCV 窗口，按上表键盘切换背景模式。

</div>

In [4]:
def add_panel(image, mode_text, fps):
    """在图像左上角绘制半透明信息面板。"""
    overlay = image.copy()
    cv2.rectangle(overlay, (18, 18), (520, 130), (22, 30, 46), -1)
    cv2.addWeighted(overlay, 0.45, image, 0.55, 0, image)
    lines = [
        "Demo.3 - Selfie Segmentation",
        f"Mode: {mode_text}    FPS: {fps:.1f}",
        "Keys: x=green  c=custom  z=original  q=quit",
    ]
    for idx, line in enumerate(lines):
        cv2.putText(
            image, line, (30, 48 + idx * 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.62,
            (255, 245, 225) if idx == 0 else (210, 225, 240),
            2, cv2.LINE_AA,
        )


def main():
    from IPython.display import clear_output, display
    from PIL import Image

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("无法打开摄像头 0。")

    def highgui_available():
        try:
            cv2.namedWindow('highgui_test', cv2.WINDOW_NORMAL)
            cv2.destroyWindow('highgui_test')
            return True
        except cv2.error:
            return False

    use_highgui = highgui_available()
    max_preview_frames = 60
    frame_counter = 0
    current_mode = "original"
    prev_time = time.time()
    cached_background = None

    if use_highgui:
        print("已启用 OpenCV 窗口模式，按 x/c/z 切换背景，按 q 或 ESC 退出。")
    else:
        print("当前环境不支持 cv2.imshow，已切换为 notebook 内联预览模式。")
        print(f"内联模式将自动预览 {max_preview_frames} 帧。")

    options = mp_vision.ImageSegmenterOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp_vision.RunningMode.VIDEO,
        output_confidence_masks=True,
    )
    with mp_vision.ImageSegmenter.create_from_options(options) as segmenter:
        while True:
            success, frame = cap.read()
            if not success:
                print("读取摄像头帧失败。")
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(time.time() * 1000)
            seg_result = segmenter.segment_for_video(mp_image, timestamp_ms)

            masks = seg_result.confidence_masks
            raw_mask = masks[-1].numpy_view() if len(masks) > 1 else masks[0].numpy_view()
            mask = cv2.GaussianBlur(raw_mask, (15, 15), 0)
            person_mask = np.where(mask > 0.45, mask, 0.0)

            if current_mode == "green":
                bg = np.full_like(frame, (0, 255, 0))
                output = composite(frame, person_mask, bg)
            elif current_mode == "custom":
                if cached_background is None or cached_background.shape[:2] != (height, width):
                    cached_background = load_background((width, height))
                output = composite(frame, person_mask, cached_background)
            else:
                output = frame.copy()

            current_time = time.time()
            fps = 1.0 / max(current_time - prev_time, 1e-6)
            prev_time = current_time
            add_panel(output, current_mode, fps)

            if use_highgui:
                cv2.imshow(WINDOW_NAME, output)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("x"):
                    current_mode = "green"
                elif key == ord("c"):
                    current_mode = "custom"
                elif key == ord("z"):
                    current_mode = "original"
                elif key in (ord("q"), 27):
                    break
            else:
                output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
                clear_output(wait=True)
                display(Image.fromarray(output_rgb))
                time.sleep(0.05)
                frame_counter += 1
                if frame_counter >= max_preview_frames:
                    print("内联预览已结束。若要继续观察，请重新运行本单元。")
                    break

    cap.release()
    if use_highgui:
        try:
            cv2.destroyAllWindows()
            cv2.waitKey(1)
        except cv2.error:
            pass


main()

已启用 OpenCV 窗口模式，按 x/c/z 切换背景，按 q 或 ESC 退出。


### 运行前，先带着比较任务去看结果

为了避免只停留在“效果挺酷”，建议运行时做 3 组比较：

1. 比较 `original` 和 `green`：哪些边缘在纯色背景下更容易暴露瑕疵？
2. 比较 `green` 和 `custom`：复杂背景为什么更容易让抠图缺陷显眼？
3. 尝试轻微晃动身体、抬手、转头，观察掩码边缘是平滑变化还是突然破碎。

如果课堂时间允许，还可以让学生主动修改阈值 `0.45` 或模糊核大小 `(15, 15)`，比较边缘质量如何变化。

### 结果解读与排错提示

如果课堂演示时抠图边缘不够自然，可以优先检查这几项：

1. 光照太暗或人物离镜头太远时，掩码边缘会变粗糙。
2. 头发丝、透明物体和快速挥动的手最容易出现误分割。
3. 自定义背景越复杂，边缘瑕疵越容易被看出来。

建议同学课后尝试：保留代码结构不变，只修改阈值、模糊核大小或背景图片，对比哪种组合看起来最自然。